## Evaluation

##  1. Evaluating Summaries (LLM-as-Judge) 

Since source is Wiki text, we have clean ground truth. Use LLM as judge given the summary and source text:
#### What Your LLM Judge Covers
* Faithfulness — does the summary only use what's in the source text?
* Coverage — given the source text available, did the summary capture the important themes? (accounts for thin wikis fairly)
* Tagline accuracy — is the tagline grounded in the source, specific to this metro, not generic?

In [20]:
import os
import re
import json
import time
import random
import boto3
import pandas as pd

In [21]:
# ── CONFIG ────────────────────────────────────────────────────────────────────

INPUT_FILE       = "../data/processed/cbsa_wiki_wikivoyage_summaries_df.csv"
OUTPUT_CSV       = "../data/evaluation/summary_eval_results.csv"

SAMPLE_SIZE      = 100    # out of 900+; stratified across small/medium/large
RANDOM_SEED      = 42
SLEEP_BETWEEN    = 0.3   # seconds between API calls — avoids rate limits

# Column names in your CSV — adjust if different
COL_CBSA         = "cbsa_name"
COL_SOURCE       = "city_wiki_wikivoyage_text"
COL_SUMMARY      = "summary"
COL_TAGLINE      = "tagline"

In [22]:
# ── LLM-AS-JUDGE ─────────────────────────────────────────────────────────────

JUDGE_PROMPT = """You are evaluating an AI-generated city summary for a relocation tool.
The summary was generated ONLY from the source wiki text below.

SOURCE TEXT (ground truth):
{source_text}

GENERATED TAGLINE:
{tagline}

GENERATED SUMMARY:
{summary}

Score on these dimensions. Be strict. Return ONLY valid JSON, no other text.

{{
  "faithfulness": <1-5>,
  "faithfulness_reason": "<one sentence>",

  "coverage": <1-5>,
  "coverage_reason": "<one sentence — are major themes from source represented?>",

  "no_fabrication": <1-5>,
  "no_fabrication_reason": "<one sentence — does summary avoid specific claims not in source?>",

  "thin_source_handled": <1-5>,
  "thin_source_reason": "<one sentence — if source is sparse, did summary stay short rather than pad?>",

  "tagline_accuracy": <1-5>,
  "tagline_reason": "<one sentence — is tagline factually grounded in the source text?>"
}}

Scoring guide:
5 = excellent, fully meets the criterion
4 = good, minor issues
3 = acceptable, noticeable issues  
2 = poor, significant problems
1 = fails criterion entirely
"""

def call_llm_judge(client , row: pd.Series) -> dict:
    # Truncate source text to keep costs low — first 3000 chars is enough for judge
    source_truncated = row[COL_SOURCE] #[:3000]

    prompt = JUDGE_PROMPT.format(
        source_text = source_truncated,
        tagline     = row[COL_TAGLINE],
        summary     = row[COL_SUMMARY],
    )

    try:

        body = json.dumps({
            "messages": [{"role": "user", "content": prompt}],
            "max_tokens": 512,
            "anthropic_version": "bedrock-2023-05-31"
        })

        response = client.invoke_model(
            modelId='anthropic.claude-3-haiku-20240307-v1:0',
            body=body
        )
    
        response_body = json.loads(response['body'].read())
        text = response_body['content'][0]['text']

        # Strip markdown fences if present
        text = re.sub(r"^```json|^```|```$", "", text, flags=re.MULTILINE).strip()
        scores = json.loads(text)
        scores["llm_judge_error"] = ""
        return scores

    except json.JSONDecodeError as e:
        return {"llm_judge_error": f"JSON parse error: {e}", "raw_response": text}
    except Exception as e:
        return {"llm_judge_error": str(e)}

In [23]:
def stratified_sample(df: pd.DataFrame, n: int, seed: int) -> pd.DataFrame:
    """
    Sample across small/medium/large CBSAs.
    Uses city_count if available, else source text length as proxy.
    """
    df = df.copy()

    size_col = df[COL_SOURCE].str.len()

    df["_size_bucket"] = pd.qcut(size_col, q=3, labels=["small", "medium", "large"])

    per_bucket = n // 3
    sampled = (
        df.groupby("_size_bucket", group_keys=False)
          .apply(lambda x: x.sample(min(len(x), per_bucket), random_state=seed))
    )

    # Top up to n if rounding left us short
    if len(sampled) < n:
        remaining = df[~df.index.isin(sampled.index)]
        extra = remaining.sample(min(n - len(sampled), len(remaining)), random_state=seed)
        sampled = pd.concat([sampled, extra])

    return sampled.drop(columns=["_size_bucket"]).reset_index(drop=True)

In [ ]:
# ── MAIN ──────────────────────────────────────────────────────────────────────

def main():
    # Load data
    print(f"Loading {INPUT_FILE}...")
    df = pd.read_csv(INPUT_FILE)

    print(f"Total CBSAs: {len(df)}")

    # Validate required columns
    for col in [COL_CBSA, COL_SOURCE, COL_SUMMARY, COL_TAGLINE]:
        assert col in df.columns, f"Missing column: {col}. Check CONFIG section."

    # Drop rows with missing values in key columns
    df = df.dropna(subset=[COL_SOURCE, COL_SUMMARY, COL_TAGLINE])
    print(f"CBSAs after dropping nulls: {len(df)}")

    # Stratified sample
    sample_df = stratified_sample(df, SAMPLE_SIZE, RANDOM_SEED)
    print(f"Sampled {len(sample_df)} CBSAs for evaluation (stratified by size)")

    # Init boto3 client
    client = boto3.client(
        service_name='bedrock-runtime',
        region_name='us-east-1'
    )


    results = []

    for i, row in sample_df.iterrows():
        print(f"[{i+1}/{len(sample_df)}] Evaluating: {row[COL_CBSA]}")
        result = {COL_CBSA: row[COL_CBSA]}
        
        # LLM judge
        llm_scores = call_llm_judge(client, row)
        result.update(llm_scores)

        results.append(result)
        time.sleep(SLEEP_BETWEEN)

    results_df = pd.DataFrame(results)
    results_df.to_csv(OUTPUT_CSV, index=False)
    print(f"\nResults saved to {OUTPUT_CSV}")

    # overalls scores
    llm_dims = ["faithfulness", "coverage", "no_fabrication", 
            "thin_source_handled", "tagline_accuracy"]

    results_df[llm_dims] = results_df[llm_dims].apply(pd.to_numeric, errors='coerce')
    results_df["overall_score"] = results_df[llm_dims].mean(axis=1)

    print(results_df[["cbsa_name", "overall_score"] + llm_dims].sort_values("overall_score"))
    print(f"\nMean overall score: {results_df['overall_score'].mean():.2f}/5")

   
main()

Loading ../data/processed/cbsa_wiki_wikivoyage_summaries_df.csv...
Total CBSAs: 935
CBSAs after dropping nulls: 932
Sampled 100 CBSAs for evaluation (stratified by size)
[1/100] Evaluating: Urbana, OH
[2/100] Evaluating: Altus, OK
[3/100] Evaluating: Cody, WY
[4/100] Evaluating: Cordele, GA
[5/100] Evaluating: Bay City, TX
[6/100] Evaluating: Corsicana, TX
[7/100] Evaluating: Fergus Falls, MN
[8/100] Evaluating: Mineral Wells, TX
[9/100] Evaluating: Russellville, AL
[10/100] Evaluating: Canton, IL
[11/100] Evaluating: Douglas, GA
[12/100] Evaluating: Logansport, IN
[13/100] Evaluating: Washington Court House, OH
[14/100] Evaluating: Yuba City, CA
[15/100] Evaluating: Troy, AL
[16/100] Evaluating: Uvalde, TX
[17/100] Evaluating: Albertville, AL
[18/100] Evaluating: Laurinburg, NC
[19/100] Evaluating: Ludington, MI
[20/100] Evaluating: Magnolia, AR
[21/100] Evaluating: Deming, NM
[22/100] Evaluating: Gainesville, FL
[23/100] Evaluating: Hutchinson, MN
[24/100] Evaluating: Oak Harbor, WA


## 2. Evaluating Retrieval (Semantic Search)

In [25]:
import sys
import os
sys.path.append(os.path.abspath(".."))
from src import semantic_search

In [26]:
test_cases = [
    {
        "query": "city with live music and arts scene",
        "expected": ["Austin-Round Rock-San Marcos, TX"]
    },
    {
        "query": "research triangle",
        "expected": ["Raleigh-Cary, NC", "Durham-Chapel Hill, NC"]
    },
    {
        "query": "technical hub",
        "expected": ["Seattle-Tacoma-Bellevue, WA"]
    },
    {
        "query": "silicon valley",
        "expected": ["San Francisco-Oakland-Fremont, CA", "San Jose-Sunnyvale-Santa Clara, CA"]
    },
    {
        "query": "automotive hub",
        "expected": ["Detroit-Warren-Dearborn, MI"]
    }
]

def evaluate_retrieval(test_cases, top_k=5):
    for test in test_cases:
        results = semantic_search.search(test["query"], top_k=top_k)
        returned_cbsas = [r["cbsa"] for r in results]

        hits = [cbsa for cbsa in test["expected"] if cbsa in returned_cbsas]
        precision = len(hits) / len(test["expected"])

        print(f"\nQuery: {test['query']}")
        print(f"Expected : {test['expected']}")
        print(f"Returned : {returned_cbsas}")
        print(f"Hits     : {hits}")
        print(f"Precision: {precision:.2f}")

evaluate_retrieval(test_cases)


Query: city with live music and arts scene
Expected : ['Austin-Round Rock-San Marcos, TX']
Returned : ['Austin-Round Rock-San Marcos, TX', 'Torrington, CT', 'Tullahoma-Manchester, TN', 'Rockford, IL', 'Kansas City, MO-KS']
Hits     : ['Austin-Round Rock-San Marcos, TX']
Precision: 1.00

Query: research triangle
Expected : ['Raleigh-Cary, NC', 'Durham-Chapel Hill, NC']
Returned : ['Arecibo, PR', 'Los Alamos, NM', 'Durham-Chapel Hill, NC', 'Raleigh-Cary, NC', 'Greensboro-High Point, NC']
Hits     : ['Raleigh-Cary, NC', 'Durham-Chapel Hill, NC']
Precision: 1.00

Query: technical hub
Expected : ['Seattle-Tacoma-Bellevue, WA']
Returned : ['Lubbock, TX', 'Seattle-Tacoma-Bellevue, WA', 'Marion-Herrin, IL', 'Effingham, IL', 'Portland-Vancouver-Hillsboro, OR-WA']
Hits     : ['Seattle-Tacoma-Bellevue, WA']
Precision: 1.00

Query: silicon valley
Expected : ['San Francisco-Oakland-Fremont, CA', 'San Jose-Sunnyvale-Santa Clara, CA']
Returned : ['San Jose-Sunnyvale-Santa Clara, CA', 'Portland-Vanco

In [27]:
test_cases = [
    {
        "query": "Coastal big city with hiking trails"
    },
    {
        "query": "Large city with good public transit"
    },
    {
        "query": "Walkable city with nightlife",
    },
    {
        "query": "Urban city with lots of parks",
    },
    {
        "query": "Mountain city with outdoor activities",
    },
    {
        "query": "good place for young professionals",
    }
]
for test in test_cases:
    results = semantic_search.search(test["query"], top_k=5)
    print(f"\nQuery: {test['query']}")
    for r in results:
        print(f"  {r['cbsa']}")


Query: Coastal big city with hiking trails
  Traverse City, MI
  Michigan City-La Porte, IN
  San Diego-Chula Vista-Carlsbad, CA
  Panama City-Panama City Beach, FL
  Crestview-Fort Walton Beach-Destin, FL

Query: Large city with good public transit
  Chicago-Naperville-Elgin, IL-IN
  San Jose-Sunnyvale-Santa Clara, CA
  Atlanta-Sandy Springs-Roswell, GA
  Detroit-Warren-Dearborn, MI
  Hagerstown-Martinsburg, MD-WV

Query: Walkable city with nightlife
  Traverse City, MI
  Michigan City-La Porte, IN
  Las Vegas-Henderson-North Las Vegas, NV
  Urban Honolulu, HI
  Adrian, MI

Query: Urban city with lots of parks
  Michigan City-La Porte, IN
  Denver-Aurora-Centennial, CO
  Dodge City, KS
  Reading, PA
  Rapid City, SD

Query: Mountain city with outdoor activities
  Mountain Home, AR
  Bozeman, MT
  Traverse City, MI
  Breckenridge, CO
  Brigham City, UT-ID

Query: good place for young professionals
  Durham-Chapel Hill, NC
  Youngstown-Warren, OH
  Greensboro-High Point, NC
  Little Ro

The system is generally performing well in identifying relevant cities and understanding user lifestyle preferences such as coastal living, mountains, transit access, walkability, and nightlife. However, it struggles to clearly distinguish between city scale and type, particularly in separating large urban metros from smaller towns or rural areas. 